# LLM Judge - RunPod Automated v2

## Overview

This notebook automates LLM-as-a-Judge evaluation using RunPod infrastructure with **parallel pod execution**.

### Key Features (from successful manual implementation)

- ✅ **Session-based state management** - Each pod tracks its own state
- ✅ **Comprehensive logging** - Per-pod logs + master aggregated log (UTF-8)
- ✅ **Progress tracking** - Before/after matrices show completion status
- ✅ **Retry logic** - Pod creation has 3 attempts with exponential backoff
- ✅ **Test-first approach** - Validate on small sample before scaling
- ✅ **Smart resumption** - Skips complete models, re-judges partials
- ✅ **Parallel execution** - Distributes work across multiple pods

### Architecture

1. **Discovery**: Build progress matrix to identify models needing judgment
2. **Test Phase**: Single pod validates pipeline with N=10 samples
3. **Distribution**: Chunk remaining work across MAX_CONCURRENT_PODS
4. **Execution**: Parallel pod provisioning and judging
5. **Aggregation**: Collect results and generate summary

### Resumption Strategy

- **Complete models**: Skipped entirely
- **Partial models**: Re-judged from scratch (overwrite)
- **Missing models**: Judged from scratch
- **Force flag**: `FORCE_REJUDGE=True` re-judges everything

---

## Section 1: Configuration & Setup

In [1]:
# Imports
import os
import sys
import json
import time
import shutil
import stat
import posixpath
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import runpod
import paramiko
import requests
import pandas as pd
from tqdm.auto import tqdm

print("✓ Imports loaded")

utils.py            :164  2025-12-04 00:51:09,757 NumExpr defaulting to 8 threads.


✓ Imports loaded


In [ ]:
# ========================================
# CONFIGURATION
# ========================================

# RunPod Settings
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]
runpod.api_key = RUNPOD_API_KEY
IMAGE_NAME = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
GPU_TYPE = "NVIDIA H200"  # Use H200 for large models like gpt-oss:120b
GPU_COUNT = 1
CONTAINER_DISK_GB = 200

# SSH Settings
SSH_KEY_PATH = str(Path.home() / ".ssh" / "id_ed25519")

# Task & Judge Configuration
TASK_NAME = "sysengbench-osq"
JUDGE_MODEL = "gpt-oss:120b"  # Ollama model for judging
PROMPT_ID = "p1"  # "p1" = scores only, "p2" = scores + justifications
TEMPERATURE = 0.0
MAX_TOKENS = 2000

# Execution Settings
MAX_CONCURRENT_PODS = 2  # Number of parallel pods

# IMPORTANT: SAMPLE_N vs TEST_SAMPLE_N Behavior
# ==============================================
# - TEST_SAMPLE_N: Number of samples for TEST PHASE (validates single model)
#   * Used ONLY in test phase (Section 5)
#   * Tests 1 model with N samples (e.g., 10 samples)
#
# - SAMPLE_N: Number of samples PER MODEL during PARALLEL EXECUTION
#   * Used in parallel execution phase (Section 6)
#   * Applies to ALL models, not just first N models!
#   * 0 = judge ALL samples (845 per model)
#   * >0 = judge first N samples per model (e.g., 50 = first 50 samples)
#   * Example: SAMPLE_N=50 with 19 models = 19 × 50 = 950 total judgments
#
SAMPLE_N = 0  # 0 = ALL samples per model, >0 = limit samples per model

# Resumption Behavior
FORCE_REJUDGE = False  # True = ignore existing outputs, re-judge everything
                       # False = skip complete models, re-do partials

# Test Phase Settings
TEST_BEFORE_FULL_RUN = True  # Run test phase before scaling up
TEST_SAMPLE_N = 10  # Number of samples for test (single model validation)
DELETE_TEST_OUTPUTS = False  # Clean up test outputs after validation

# Paths
BASE_DIR = Path.cwd()
PHASE4_ROOT = BASE_DIR.parent / "phase4_inference" / "output"
# PHASE5_ROOT = BASE_DIR / f"{TASK_NAME}-llm-judge"
PHASE5_ROOT = BASE_DIR / f"{TASK_NAME}-llm-judge-test"
LOG_DIR = BASE_DIR / "runpod_llm_judge_logs"

# Create directories
PHASE5_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("=== Configuration ===")
print(f"RunPod API Key present: {bool(RUNPOD_API_KEY)}")
print(f"GPU: {GPU_TYPE} x {GPU_COUNT}")
print(f"Task: {TASK_NAME}")
print(f"Judge Model: {JUDGE_MODEL}")
print(f"Prompt ID: {PROMPT_ID}")
print(f"Max Concurrent Pods: {MAX_CONCURRENT_PODS}")
print(f"\n📊 Sample Configuration:")
print(f"  TEST_SAMPLE_N: {TEST_SAMPLE_N} (test phase only)")
print(f"  SAMPLE_N: {SAMPLE_N if SAMPLE_N > 0 else 'ALL'} (per model in parallel execution)")
print(f"Force Re-judge: {FORCE_REJUDGE}")
print(f"Test Phase: {TEST_BEFORE_FULL_RUN}")
print(f"\nPaths:")
print(f"  Phase 4 Root: {PHASE4_ROOT}")
print(f"  Phase 5 Root: {PHASE5_ROOT}")
print(f"  Log Directory: {LOG_DIR}")

In [3]:
# Judge Prompts (from manual implementation)

# Prompt 1: Scores only
JUDGE_PROMPT_P1 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>}},
  "conceptual_understanding": {{"score": <0–20>}},
  "completeness": {{"score": <0–20>}},
  "clarity_organization": {{"score": <0–20>}},
  "professional_relevance": {{"score": <0–20>}},
  "overall_score": <0–100>
}}
"""

# Prompt 2: Scores + justifications
JUDGE_PROMPT_P2 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "conceptual_understanding": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "completeness": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "clarity_organization": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "professional_relevance": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}}
"""

# Select prompt based on PROMPT_ID
JUDGE_PROMPT = JUDGE_PROMPT_P1 if PROMPT_ID == "p1" else JUDGE_PROMPT_P2

print(f"✓ Loaded judge prompt: {PROMPT_ID}")

✓ Loaded judge prompt: p1


## Section 2: Helper Functions & PodSession Class

In [ ]:
# Helper Functions

def create_session_logger(log_file: Path):
    """Create a logger function for a session."""
    def log(msg: str, level: str = 'INFO'):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        formatted = f"[{timestamp}] [{level}] {msg}"
        print(formatted)
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(formatted + '\n')
    return log


def create_pod_with_retries(log_func, max_attempts=3, retry_delay=10, **kwargs):
    """
    Attempt to create a RunPod pod with retries.
    Returns the created pod dict.
    Raises RuntimeError after max failures.
    """
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            log_func(f"Attempt {attempt}/{max_attempts}: creating pod...")
            pod = runpod.create_pod(**kwargs)
            pod_id = pod.get("id")
            log_func(f"Pod created: {pod_id}")
            return pod
        except Exception as e:
            last_err = e
            log_func(f"⚠ Pod creation failed on attempt {attempt}: {e}")
            if attempt < max_attempts:
                log_func(f"Waiting {retry_delay} seconds before retrying...")
                time.sleep(retry_delay)
    
    raise RuntimeError(f"❌ Failed to create pod after {max_attempts} attempts. Last error: {last_err}")


def run_and_check(ssh, cmd, desc, log_func):
    """Run a remote command and wait for completion."""
    log_func(f"▶ {desc}")
    stdin, stdout, stderr = ssh.exec_command(cmd)
    log_func(f"[DEBUG] Command sent to remote pod: {cmd}")

    out = stdout.read().decode()
    err = stderr.read().decode()

    exit_code = stdout.channel.recv_exit_status()
    log_func(f"[DEBUG] Exit code received: {exit_code}")
    # out = stdout.read().decode()
    # err = stderr.read().decode()
    if out:
        log_func(out)
    if err:
        log_func(f"stderr: {err}")
    if exit_code != 0:
        raise RuntimeError(f"{desc} failed with exit code {exit_code}")
    log_func(f"✔ {desc} finished.")
    return out


def stream_ssh_command(ssh, cmd, desc, log_func):
    """
    Run a remote command with real-time output streaming.
    Shows progress as it happens instead of blocking until completion.
    """
    log_func(f"▶ {desc}")
    log_func(f"[DEBUG] Command: {cmd}")

    # Get SSH transport channel
    chan = ssh.get_transport().open_session()
    chan.exec_command(cmd)

    log_func(f"[STREAMING OUTPUT]")
    log_func("=" * 60)

    # Stream output in real-time
    while True:
        # Check for stdout
        if chan.recv_ready():
            data = chan.recv(4096).decode('utf-8', errors='replace')
            for line in data.splitlines():
                if line.strip():
                    output_line = f"  {line}"
                    print(output_line)
                    log_func(output_line)

        # Check for stderr
        if chan.recv_stderr_ready():
            data = chan.recv_stderr(4096).decode('utf-8', errors='replace')
            for line in data.splitlines():
                if line.strip():
                    error_line = f"  [stderr] {line}"
                    print(error_line)
                    log_func(error_line, level='WARN')

        # Check if command finished
        if chan.exit_status_ready():
            break

        time.sleep(0.1)  # Small delay to avoid CPU spinning

    # Get final exit code
    exit_code = chan.recv_exit_status()

    log_func("=" * 60)
    log_func(f"[DEBUG] Exit code: {exit_code}")

    if exit_code != 0:
        raise RuntimeError(f"{desc} failed with exit code {exit_code}")

    log_func(f"✔ {desc} finished successfully")

    chan.close()
    return exit_code


def download_dir(sftp, remote_dir, local_dir, log_func):
    """Recursively download a directory from remote pod."""
    try:
        entries = sftp.listdir_attr(remote_dir)
    except IOError:
        log_func(f"Remote dir {remote_dir} doesn't exist")
        return
    
    local_dir.mkdir(parents=True, exist_ok=True)
    
    for entry in entries:
        remote_path = posixpath.join(remote_dir, entry.filename)
        local_path = local_dir / entry.filename
        
        if stat.S_ISDIR(entry.st_mode):
            download_dir(sftp, remote_path, local_path, log_func)
        else:
            log_func(f"Downloading {remote_path} -> {local_path}")
            sftp.get(remote_path, str(local_path))


print("✓ Helper functions loaded (with streaming SSH support)")

In [ ]:
# PodSession Class

class PodSession:
    """Encapsulates a single pod's lifecycle with state management."""
    
    def __init__(self, pod_id: int, log_file: Path, models: List[str]):
        self.pod_id = pod_id
        self.log_file = log_file
        self.models = models
        self.log = create_session_logger(log_file)
        
        # State
        self.runpod_id: Optional[str] = None
        self.ssh: Optional[paramiko.SSHClient] = None
        self.ssh_host: Optional[str] = None
        self.ssh_port: Optional[int] = None
        self.start_time = time.time()
        self.results = []
        
        self.log(f"=== Pod Session {pod_id} Initialized ===")
        self.log(f"Models assigned: {len(models)}")
    
    def create_pod(self):
        """Create RunPod instance."""
        self.log("Creating RunPod instance...")
        pod = create_pod_with_retries(
            self.log,
            name=f"llm-judge-auto-v2-{self.pod_id}",
            image_name=IMAGE_NAME,
            gpu_type_id=GPU_TYPE,
            gpu_count=GPU_COUNT,
            container_disk_in_gb=CONTAINER_DISK_GB,
            support_public_ip=True,
            ports="22/tcp,11434/tcp"
        )
        self.runpod_id = pod.get("id")
        self.log(f"RunPod created: {self.runpod_id}")
    
    def wait_for_ssh(self, max_attempts=60, poll_interval=10):
        """Wait for SSH port to become available."""
        self.log(f"Waiting for SSH port (max {max_attempts * poll_interval}s)...")
        
        for attempt in range(max_attempts):
            try:
                pod_info = runpod.get_pod(self.runpod_id)
                runtime = pod_info.get("runtime", {})
                ports = runtime.get("ports", [])
                
                for port_info in ports:
                    if port_info.get("privatePort") == 22:
                        self.ssh_host = port_info.get("ip")
                        self.ssh_port = port_info.get("publicPort")
                        self.log(f"SSH available: {self.ssh_host}:{self.ssh_port}")
                        return
            except Exception as e:
                self.log(f"Attempt {attempt + 1}/{max_attempts}: {e}")
            
            time.sleep(poll_interval)
        
        raise RuntimeError(f"SSH port not available after {max_attempts * poll_interval}s")
    
    def connect_ssh(self):
        """Establish SSH connection."""
        self.log("Connecting via SSH...")
        self.ssh = paramiko.SSHClient()
        self.ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        private_key = paramiko.Ed25519Key.from_private_key_file(SSH_KEY_PATH)
        
        self.ssh.connect(
            hostname=self.ssh_host,
            port=self.ssh_port,
            username='root',
            pkey=private_key,
            timeout=30
        )
        self.log("SSH connected successfully")
    
    def provision_pod(self):
        """Provision the pod with Ollama and judge model."""
        self.log("Starting pod provisioning...")
        
        # Update apt and install dependencies
        run_and_check(self.ssh, "apt-get update -y", "Update apt", self.log)
        
        # Install Ollama
        self.log("Installing Ollama...")
        run_and_check(
            self.ssh,
            "curl -fsSL https://ollama.com/install.sh | sh",
            "Install Ollama",
            self.log
        )
        
        # Start Ollama server in background
        self.log("Starting Ollama server...")
        self.ssh.exec_command("nohup ollama serve > /tmp/ollama.log 2>&1 &")
        time.sleep(5)  # Let server start
        
        # Test Ollama API
        self.log("Testing Ollama API...")
        run_and_check(
            self.ssh,
            "curl -s http://localhost:11434/api/tags",
            "Test Ollama API",
            self.log
        )
        
        # Pull judge model
        self.log(f"Pulling judge model: {JUDGE_MODEL}...")
        self.log("⏳ This may take several minutes for large models...")
        run_and_check(
            self.ssh,
            f"ollama pull {JUDGE_MODEL}",
            f"Pull {JUDGE_MODEL}",
            self.log
        )
        
        # Verify model
        out = run_and_check(
            self.ssh,
            "ollama list",
            "List models",
            self.log
        )
        if JUDGE_MODEL in out:
            self.log(f"✓ Judge model {JUDGE_MODEL} is ready")
        else:
            self.log(f"⚠ Warning: {JUDGE_MODEL} not found in model list", level='WARN')
        
        self.log("✓ Pod provisioning complete")
    
    def upload_worker_script(self, script_content: str):
        """Upload worker script to pod."""
        self.log("Uploading worker script...")
        sftp = self.ssh.open_sftp()
        with sftp.file('/root/worker.py', 'w') as f:
            f.write(script_content)
        sftp.close()
        self.log("✓ Worker script uploaded")
    
    def judge_model(self, model_name: str, input_data_path: Path, is_test: bool = False) -> Dict:
        """Judge a single model."""
        self.log(f"{'[TEST] ' if is_test else ''}Judging model: {model_name}")
        
        # Upload input data
        remote_input = f"/root/input_{model_name.replace(':', '_')}.jsonl"
        remote_output = f"/root/output_{model_name.replace(':', '_')}.jsonl"
        
        sftp = self.ssh.open_sftp()
        sftp.put(str(input_data_path), remote_input)
        sftp.close()
        self.log(f"  Uploaded input: {input_data_path.name}")
        
        
        # Run worker script
        cmd = (
            f"python3 /root/worker.py "
            f"--input {remote_input} "
            f"--output {remote_output} "
            f"--judge-model {JUDGE_MODEL} "
            f"--prompt-id {PROMPT_ID} "
            f"--temperature {TEMPERATURE} "
            f"--max-tokens {MAX_TOKENS} "
            f"--task-name {TASK_NAME} "
            f"--model-name {model_name}"
        )

        self.log(f"[DEBUG] Starting worker with command: {cmd}")
        self.log(f"  Running worker script with real-time streaming...")
        self.log("=" * 60)

        start = time.time()

        # Use streaming to show real-time progress
        stream_ssh_command(self.ssh, cmd, f"Judge {model_name}", self.log)

        elapsed = time.time() - start
        self.log("=" * 60)
        self.log(f"[DEBUG] Worker finished for {model_name}")
        self.log(f"  ✓ Judging complete ({elapsed:.1f}s)")
        
        # Determine output directory
        if not is_test:
            output_dir = PHASE5_ROOT / model_name
        else:
            output_dir = PHASE5_ROOT / "test_outputs" / model_name

        output_dir.mkdir(parents=True, exist_ok=True)

        # Extract Phase 4 timestamp from the input filename
        phase4_filename = input_data_path.name  # e.g. samples_sysengbench_2025-11-16T04-31-27.456510.jsonl
        phase4_timestamp = phase4_filename.split("_")[-1].replace(".jsonl", "")

        # Build output filename that matches Phase 4
        safe_model = model_name.replace(":", "_")
        judge_suffix = JUDGE_MODEL.replace(":", "_")

        local_output = output_dir / (
            f"samples_{TASK_NAME}_{phase4_timestamp}__{judge_suffix}-{PROMPT_ID}.jsonl"
        )

        # Download output to model-specific folder
        sftp = self.ssh.open_sftp()
        sftp.get(remote_output, str(local_output))
        sftp.close()

        self.log(f"  Downloaded: {local_output}")


        # # Download output
        # if not is_test:
        #     output_dir = PHASE5_ROOT / model_name
        # else:
        #     output_dir = PHASE5_ROOT / "test_outputs" / model_name
        
        # output_dir.mkdir(parents=True, exist_ok=True)
        # judge_suffix = JUDGE_MODEL.replace(":", "_")

        # local_output = output_dir / f"samples_{TASK_NAME}__{judge_suffix}-{PROMPT_ID}.jsonl"
        
        # sftp = self.ssh.open_sftp()
        # sftp.get(remote_output, str(local_output))
        # sftp.close()
        # self.log(f"  Downloaded: {local_output}")
        
        # Count results
        with open(local_output, 'r', encoding='utf-8') as f:
            count = sum(1 for _ in f)
        
        result = {
            'model_name': model_name,
            'output_file': str(local_output),
            'sample_count': count,
            'elapsed_s': elapsed
        }
        self.results.append(result)
        
        return result
    
    def cleanup(self):
        """Close SSH and terminate pod."""
        self.log("Cleaning up...")
        
        if self.ssh:
            self.ssh.close()
            self.log("SSH connection closed")
        
        if self.runpod_id:
            try:
                runpod.terminate_pod(self.runpod_id)
                self.log(f"Pod {self.runpod_id} terminated")
            except Exception as e:
                self.log(f"Error terminating pod: {e}", level='ERROR')
        
        elapsed = time.time() - self.start_time
        self.log(f"=== Pod Session Complete ({elapsed:.1f}s) ===")
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.cleanup()


print("✓ PodSession class loaded (with streaming support)")

## Section 3: Worker Script

In [ ]:
# Pod Worker Script - VERBOSE VERSION with Real-Time Progress

POD_WORKER_SCRIPT = '''import json
import requests
import sys
import time
from datetime import datetime

JUDGE_PROMPT_P1 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{
  "technical_accuracy": {"score": <0–20>},
  "conceptual_understanding": {"score": <0–20>},
  "completeness": {"score": <0–20>},
  "clarity_organization": {"score": <0–20>},
  "professional_relevance": {"score": <0–20>},
  "overall_score": <0–100>
}
"""

JUDGE_PROMPT_P2 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{
  "technical_accuracy": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "conceptual_understanding": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "completeness": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "clarity_organization": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "professional_relevance": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}
"""


def safe_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None


def extract_student_response(sample_row):
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""


def derive_prompt_fields(phase4_row):
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level = doc.get("blooms_level", "N/A")
    se_domain = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp = extract_student_response(phase4_row)
    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }


def triad_missing(fields):
    return [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()
    ]


def run_worker(input_path, output_path, judge_model, prompt_id,
               temperature, max_tokens, task_name, model_name):
    judge_prompt = JUDGE_PROMPT_P1 if prompt_id == "p1" else JUDGE_PROMPT_P2
    judge_url = "http://localhost:11434/v1/chat/completions"

    # Count total samples first
    with open(input_path, "r", encoding="utf-8") as fin:
        total_samples = sum(1 for line in fin if line.strip())

    print(f"[worker] Starting judging for {model_name}", flush=True)
    print(f"[worker] Total samples to judge: {total_samples}", flush=True)
    print(f"[worker] Judge model: {judge_model}", flush=True)
    print(f"[worker] Prompt ID: {prompt_id}", flush=True)
    print("=" * 60, flush=True)

    current = 0
    start_time = time.time()

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            if not line.strip():
                continue

            current += 1
            sample_start = time.time()

            payload = json.loads(line)
            sid = payload["sample_id"]
            phase4_row = payload["phase4_row"]

            # Progress update every 10 samples or first/last
            if current % 10 == 0 or current == 1 or current == total_samples:
                elapsed = time.time() - start_time
                rate = current / elapsed if elapsed > 0 else 0
                remaining = (total_samples - current) / rate if rate > 0 else 0
                print(
                    f"[worker] Progress: {current}/{total_samples} "
                    f"({100*current/total_samples:.1f}%) | "
                    f"Rate: {rate:.2f} samples/s | "
                    f"ETA: {remaining/60:.1f}min",
                    flush=True
                )

            fields_for_prompt = derive_prompt_fields(phase4_row)
            missing = triad_missing(fields_for_prompt)
            if missing:
                record = {
                    "sample_id": sid,
                    "phase4_row": phase4_row,
                    "judge": {
                        "fields": None,
                        "prompt": None,
                        "raw_output": None,
                        "error": f"missing_fields: {missing}",
                        "timestamp": datetime.now().isoformat(),
                        "meta": {
                            "judge_model": judge_model,
                            "temperature": temperature,
                            "max_tokens": max_tokens,
                            "task_name": task_name,
                            "model_name": model_name,
                            "prompt_id": prompt_id,
                        },
                    },
                }
                fout.write(json.dumps(record) + "\\n")
                fout.flush()
                continue

            prompt = judge_prompt
            for key, val in fields_for_prompt.items():
                prompt = prompt.replace(f"{{{key}}}", val)

            raw = None
            parsed = None
            error = None

            try:
                response = requests.post(
                    judge_url,
                    headers={"Content-Type": "application/json"},
                    json={
                        "model": judge_model,
                        "messages": [
                            {
                                "role": "system",
                                "content": (
                                    "You are an expert evaluator in "
                                    "systems engineering education."
                                ),
                            },
                            {"role": "user", "content": prompt},
                        ],
                        "temperature": temperature,
                        "max_tokens": max_tokens,
                    },
                    timeout=120,
                )
                data = response.json()
                raw = data["choices"][0]["message"]["content"]
                parsed = safe_json(raw)
            except Exception as e:
                error = str(e)
                print(f"[worker] ERROR on sample {current}: {error}", flush=True)

            if prompt_id == "p1":
                fields = {
                    "technical_accuracy": {"score": None},
                    "conceptual_understanding": {"score": None},
                    "completeness": {"score": None},
                    "clarity_organization": {"score": None},
                    "professional_relevance": {"score": None},
                    "overall_score": None,
                }
                if parsed:
                    fields["technical_accuracy"]["score"] = (
                        (parsed.get("technical_accuracy") or {}).get("score")
                    )
                    fields["conceptual_understanding"]["score"] = (
                        (parsed.get("conceptual_understanding") or {}).get("score")
                    )
                    fields["completeness"]["score"] = (
                        (parsed.get("completeness") or {}).get("score")
                    )
                    fields["clarity_organization"]["score"] = (
                        (parsed.get("clarity_organization") or {}).get("score")
                    )
                    fields["professional_relevance"]["score"] = (
                        (parsed.get("professional_relevance") or {}).get("score")
                    )
                    fields["overall_score"] = parsed.get("overall_score")
            else:
                fields = {
                    "technical_accuracy": {"score": None, "justification": None},
                    "conceptual_understanding": {"score": None, "justification": None},
                    "completeness": {"score": None, "justification": None},
                    "clarity_organization": {"score": None, "justification": None},
                    "professional_relevance": {"score": None, "justification": None},
                    "overall_score": None,
                    "overall_assessment": None,
                    "key_strengths": None,
                    "improvement_areas": None,
                }
                if parsed:
                    for key in [
                        "technical_accuracy",
                        "conceptual_understanding",
                        "completeness",
                        "clarity_organization",
                        "professional_relevance",
                    ]:
                        fields[key] = {
                            "score": (parsed.get(key) or {}).get("score"),
                            "justification": (parsed.get(key) or {}).get("justification"),
                        }
                    fields["overall_score"] = parsed.get("overall_score")
                    fields["overall_assessment"] = parsed.get("overall_assessment")
                    fields["key_strengths"] = parsed.get("key_strengths")
                    fields["improvement_areas"] = parsed.get("improvement_areas")

            record = {
                "sample_id": sid,
                "phase4_row": phase4_row,
                "judge": {
                    "fields": fields,
                    "prompt": prompt,
                    "raw_output": raw,
                    "error": error,
                    "timestamp": datetime.now().isoformat(),
                    "meta": {
                        "judge_model": judge_model,
                        "temperature": temperature,
                        "max_tokens": max_tokens,
                        "task_name": task_name,
                        "model_name": model_name,
                        "prompt_id": prompt_id,
                    },
                },
            }
            fout.write(json.dumps(record) + "\\n")
            fout.flush()

            sample_elapsed = time.time() - sample_start
            if sample_elapsed > 10:  # Log slow samples
                print(f"[worker] Sample {current} took {sample_elapsed:.1f}s", flush=True)

    total_elapsed = time.time() - start_time
    print("=" * 60, flush=True)
    print(f"[worker] ✓ Judging complete for {model_name}", flush=True)
    print(f"[worker] Total: {total_samples} samples in {total_elapsed/60:.1f} minutes", flush=True)
    print(f"[worker] Average rate: {total_samples/total_elapsed:.2f} samples/s", flush=True)


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--judge-model", required=True)
    parser.add_argument("--prompt-id", required=True)
    parser.add_argument("--temperature", type=float, default=0.0)
    parser.add_argument("--max-tokens", type=int, default=2000)
    parser.add_argument("--task-name", required=True)
    parser.add_argument("--model-name", required=True)
    args = parser.parse_args()

    run_worker(
        input_path=args.input,
        output_path=args.output,
        judge_model=args.judge_model,
        prompt_id=args.prompt_id,
        temperature=args.temperature,
        max_tokens=args.max_tokens,
        task_name=args.task_name,
        model_name=args.model_name,
    )
'''

print(f"✓ VERBOSE worker script loaded ({len(POD_WORKER_SCRIPT)} chars)")

## Section 4: Progress Matrix & Model Discovery

In [7]:
import pandas as pd
import re

def build_multi_judge_progress_matrix(
    phase4_root: Path,
    phase5_root: Path,
    task_name: str,
    judge_model: str,
    prompt_id: str
) -> pd.DataFrame:
    """Build progress matrix to identify which models need judging."""
    p4 = phase4_root / task_name
    p5 = phase5_root

    if not p4.exists():
        raise FileNotFoundError(f"Phase 4 directory missing: {p4}")

    model_folders = sorted([d.name for d in p4.iterdir() if d.is_dir()])
    judge_suffix = judge_model.replace(":", "_")
    target_pattern = re.compile(rf"__{judge_suffix}-{prompt_id}\.jsonl$")

    rows = []

    for model in model_folders:
        model_p4_dir = p4 / model
        model_p5_dir = p5 / model

        # Total samples from Phase 4
        sample_files = [
            f for f in model_p4_dir.iterdir()
            if f.name.startswith(f"samples_{task_name}_") and f.suffix == ".jsonl"
        ]

        total_samples = 0
        if sample_files:
            sf = sample_files[0]
            with open(sf, "r", encoding="utf-8") as fh:
                total_samples = sum(1 for _ in fh)

        # Check for existing judgments
        judged_count = 0
        if model_p5_dir.exists():
            for jf in model_p5_dir.iterdir():
                if jf.is_file() and target_pattern.search(jf.name):
                    with open(jf, "r", encoding="utf-8") as fh:
                        judged_count = sum(1 for _ in fh)
                    break

        if total_samples == 0:
            status = "no_samples"
        elif judged_count == 0:
            status = "not_started"
        elif judged_count < total_samples:
            status = "partial"
        else:
            status = "complete"

        rows.append({
            "model_name": model,
            "ollama_name": model.replace("__", ":"),
            "judged_count": judged_count,
            "total_samples": total_samples,
            "progress_fraction": judged_count / total_samples if total_samples > 0 else 0.0,
            "status": status,
        })

    df = pd.DataFrame(rows)
    return df.sort_values("model_name")

print("✓ Progress matrix function loaded")

✓ Progress matrix function loaded


In [ ]:
# Build progress matrix
print("\n=== INITIAL JUDGING PROGRESS MATRIX ===\n")

progress_df = build_multi_judge_progress_matrix(
    PHASE4_ROOT,
    PHASE5_ROOT,
    TASK_NAME,
    JUDGE_MODEL,
    PROMPT_ID
)

display(progress_df)

# Categorize models
complete_models = progress_df[progress_df["status"] == "complete"]["model_name"].tolist()
partial_models = progress_df[progress_df["status"] == "partial"]["model_name"].tolist()
not_started_models = progress_df[progress_df["status"] == "not_started"]["model_name"].tolist()
no_sample_models = progress_df[progress_df["status"] == "no_samples"]["model_name"].tolist()

print(f"\n✓ Complete: {len(complete_models)} models")
if complete_models[:3]:  # Show first 3
    print(f"  Examples: {', '.join(complete_models[:3])}")

print(f"\n⚠ Partial: {len(partial_models)} models (will RE-JUDGE from scratch)")
for model in partial_models:
    row = progress_df[progress_df['model_name'] == model].iloc[0]
    print(f"  - {model}: {row['judged_count']}/{row['total_samples']} samples")

print(f"\n○ Not Started: {len(not_started_models)} models")
if not_started_models[:3]:
    print(f"  Examples: {', '.join(not_started_models[:3])}")

if no_sample_models:
    print(f"\n⊘ No Samples: {len(no_sample_models)} models (skipping)")

# Determine work queue
if FORCE_REJUDGE:
    models_to_judge = not_started_models + partial_models + complete_models
    print(f"\n🔄 FORCE_REJUDGE=True: Re-judging ALL {len(models_to_judge)} models")
else:
    models_to_judge = not_started_models + partial_models
    print(f"\n→ Work for this run: {len(models_to_judge)} models")

# NOTE: SAMPLE_N controls samples PER MODEL, not which models to process
# All models in models_to_judge will be processed, but each will only judge SAMPLE_N samples
if SAMPLE_N > 0:
    print(f"\n📊 SAMPLE_N={SAMPLE_N}: Each model will judge {SAMPLE_N} samples (not all {progress_df.iloc[0]['total_samples'] if len(progress_df) > 0 else 845})")
    print(f"   → Total judgments: {len(models_to_judge)} models × {SAMPLE_N} samples = {len(models_to_judge) * SAMPLE_N} judgments")
else:
    total_per_model = progress_df.iloc[0]['total_samples'] if len(progress_df) > 0 else 845
    print(f"\n📊 SAMPLE_N=0: Each model will judge ALL samples ({total_per_model} per model)")
    print(f"   → Total judgments: {len(models_to_judge)} models × {total_per_model} samples = {len(models_to_judge) * total_per_model} judgments")

print(f"\n→ Final work queue: {len(models_to_judge)} models (processing ALL models)")
print(f"\nModels to judge: {models_to_judge}")

In [9]:
def prepare_model_input_data(model_name: str, sample_limit: int = 0, is_test: bool = False) -> Path:
    """Prepare input JSONL for a model from Phase 4 outputs."""
    p4_dir = PHASE4_ROOT / TASK_NAME / model_name
    sample_files = [
        f for f in p4_dir.iterdir()
        if f.name.startswith(f"samples_{TASK_NAME}_") and f.suffix == ".jsonl"
    ]
    
    if not sample_files:
        raise FileNotFoundError(f"No sample files found for {model_name}")
    
    input_file = sample_files[0]
    
    # Create temporary file for judging
    if is_test:
        temp_dir = Path("/tmp/llm_judge_test")
    else:
        temp_dir = Path("/tmp/llm_judge_inputs")
    
    temp_dir.mkdir(parents=True, exist_ok=True)
    output_file = temp_dir / f"{model_name.replace(':', '_')}.jsonl"
    
    # Read and format for judging
    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:
        
        for i, line in enumerate(fin):
            if sample_limit > 0 and i >= sample_limit:
                break
            
            if not line.strip():
                continue
            
            phase4_row = json.loads(line)
            
            # Create input record for worker
            record = {
                "sample_id": i,
                "phase4_row": phase4_row
            }
            
            fout.write(json.dumps(record) + '\n')
    
    return output_file

print("✓ Input data preparation function loaded")

✓ Input data preparation function loaded


## Section 5: Test Phase

Validate the pipeline with a small sample before scaling up to multiple pods.

In [10]:
if TEST_BEFORE_FULL_RUN and len(models_to_judge) > 0:
    print("\n=== TEST PHASE ===")
    print(f"Testing with first model: {models_to_judge[0]}")
    print(f"Sample size: {TEST_SAMPLE_N}\n")
    
    # Create test log
    test_session_dir = LOG_DIR / f"test_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    test_session_dir.mkdir(parents=True, exist_ok=True)
    test_log_file = test_session_dir / "test_pod.log"
    
    test_model = models_to_judge[0]
    
    try:
        with PodSession(pod_id=0, log_file=test_log_file, models=[test_model]) as pod:
            # 1. Create pod
            pod.create_pod()
            
            # 2. Wait for SSH
            pod.wait_for_ssh()
            
            # 3. Connect SSH
            pod.connect_ssh()
            
            # 4. Provision pod
            pod.provision_pod()
            
            # 5. Upload worker script
            pod.upload_worker_script(POD_WORKER_SCRIPT)
            
            # 6. Prepare test data
            print(f"\nPreparing test data for {test_model}...")
            test_input = prepare_model_input_data(test_model, sample_limit=TEST_SAMPLE_N, is_test=True)
            print(f"Test input ready: {test_input}")
            
            # 7. Run judging
            print(f"\nRunning test judging...")
            result = pod.judge_model(test_model, test_input, is_test=True)
            
            print(f"\n=== TEST RESULTS ===")
            print(f"Model: {result['model_name']}")
            print(f"Samples judged: {result['sample_count']}")
            print(f"Time: {result['elapsed_s']:.1f}s")
            print(f"Output: {result['output_file']}")
            
            # 8. Validate output
            print(f"\nValidating output structure...")
            with open(result['output_file'], 'r', encoding='utf-8') as f:
                test_records = [json.loads(line) for line in f if line.strip()]
            
            print(f"✓ Loaded {len(test_records)} records")
            
            # Check first record structure
            if test_records:
                sample_record = test_records[0]
                print(f"\nSample record structure:")
                print(f"  - sample_id: {sample_record.get('sample_id')}")
                print(f"  - phase4_row: {'present' if sample_record.get('phase4_row') else 'missing'}")
                print(f"  - judge: {'present' if sample_record.get('judge') else 'missing'}")
                
                if sample_record.get('judge'):
                    judge = sample_record['judge']
                    print(f"  - judge.fields: {'present' if judge.get('fields') else 'missing'}")
                    print(f"  - judge.error: {judge.get('error', 'None')}")
                    
                    if judge.get('fields') and judge['fields'].get('overall_score'):
                        print(f"  - Sample score: {judge['fields']['overall_score']}")
            
            print(f"\n✓ Test phase complete!")
            print(f"\nTest log: {test_log_file}")
            
            # Cleanup test outputs if configured
            if DELETE_TEST_OUTPUTS:
                test_output_dir = PHASE5_ROOT / "test_outputs"
                if test_output_dir.exists():
                    shutil.rmtree(test_output_dir)
                    print(f"\n✓ Test outputs cleaned up")
            
            print(f"\n→ Ready to proceed with full run")
            
    except Exception as e:
        print(f"\n❌ Test phase failed: {e}")
        print(f"\nCheck test log: {test_log_file}")
        raise
        
else:
    if not TEST_BEFORE_FULL_RUN:
        print("\n⚠ TEST_BEFORE_FULL_RUN=False: Skipping test phase")
    elif len(models_to_judge) == 0:
        print("\n✓ No models to judge - skipping test phase")


=== TEST PHASE ===
Testing with first model: anthropic__claude-sonnet-4.5
Sample size: 10

[2025-12-04 00:24:31] [INFO] === Pod Session 0 Initialized ===
[2025-12-04 00:24:31] [INFO] Models assigned: 1
[2025-12-04 00:24:31] [INFO] Creating RunPod instance...
[2025-12-04 00:24:31] [INFO] Attempt 1/3: creating pod...
raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'xlydmpf7tauina', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'r15m86n7fnda', 'machine': {'podHostId': 'xlydmpf7tauina-6441198a'}}}}
[2025-12-04 00:24:31] [INFO] Pod created: xlydmpf7tauina
[2025-12-04 00:24:31] [INFO] RunPod created: xlydmpf7tauina
[2025-12-04 00:24:31] [INFO] Waiting for SSH port (max 600s)...
[2025-12-04 00:24:32] [INFO] Attempt 1/60: 'NoneType' object has no attribute 'get'
[2025-12-04 00:24:42] [INFO] Attempt 2/60:

## Section 6: Parallel Execution

Distribute work across multiple pods and execute in parallel.

In [10]:
def distribute_work(models: List[str], num_pods: int) -> List[List[str]]:
    """Distribute models across pods using round-robin."""
    if not models:
        return []
    
    # Create buckets for each pod
    buckets = [[] for _ in range(num_pods)]
    
    # Round-robin distribution
    for i, model in enumerate(models):
        bucket_idx = i % num_pods
        buckets[bucket_idx].append(model)
    
    # Filter out empty buckets
    return [b for b in buckets if b]

print("✓ Work distribution function loaded")

✓ Work distribution function loaded


In [ ]:
def run_pod_worker(pod_id: int, models: List[str], session_dir: Path) -> Dict:
    """Worker function for a single pod."""
    log_file = session_dir / f"pod_{pod_id}.log"
    
    results = {
        'pod_id': pod_id,
        'models': models,
        'success': False,
        'results': [],
        'error': None,
        'elapsed_s': 0
    }
    
    start_time = time.time()
    
    try:
        with PodSession(pod_id=pod_id, log_file=log_file, models=models) as pod:
            # Provision pod
            pod.create_pod()
            pod.wait_for_ssh()
            pod.connect_ssh()
            pod.provision_pod()
            pod.upload_worker_script(POD_WORKER_SCRIPT)
            
            # Judge each assigned model
            for model_name in models:
                try:
                    # Prepare input data
                    # IMPORTANT: SAMPLE_N limits samples per model, not model count!
                    input_data = prepare_model_input_data(
                        model_name,
                        sample_limit=SAMPLE_N,  # Apply SAMPLE_N here!
                        is_test=False
                    )

                    # Log what we're doing
                    sample_count = SAMPLE_N if SAMPLE_N > 0 else "ALL"
                    pod.log(f"  Processing {model_name} with {sample_count} samples")

                    # Run judging
                    result = pod.judge_model(model_name, input_data, is_test=False)
                    results['results'].append(result)

                    # CHECKPOINT: Log successful completion
                    pod.log(f"✓ CHECKPOINT: {model_name} completed successfully")
                    pod.log(f"  Output saved: {result['output_file']}")
                    pod.log(f"  Samples judged: {result['sample_count']}")
                    pod.log(f"  Time taken: {result['elapsed_s']:.1f}s")
                    remaining = len(models) - (models.index(model_name) + 1)
                    pod.log(f"  Models remaining in this pod: {remaining}")
                    
                except Exception as e:
                    pod.log(f"Error judging {model_name}: {e}", level='ERROR')
                    results['results'].append({
                        'model_name': model_name,
                        'error': str(e)
                    })
            
            results['success'] = True
            
    except Exception as e:
        results['error'] = str(e)
        print(f"Pod {pod_id} failed: {e}")
    
    results['elapsed_s'] = time.time() - start_time
    
    return results

print("✓ Pod worker function loaded (with SAMPLE_N support and checkpointing)")

In [ ]:
if len(models_to_judge) > 0:
    print("\n=== PARALLEL EXECUTION ===")
    
    # Create session directory
    session_dir = LOG_DIR / f"auto_v2_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    session_dir.mkdir(parents=True, exist_ok=True)
    print(f"Session directory: {session_dir}")
    
    # Distribute work
    work_distribution = distribute_work(models_to_judge, MAX_CONCURRENT_PODS)
    actual_pods = len(work_distribution)
    
    print(f"\nWork Distribution:")
    for i, models in enumerate(work_distribution):
        print(f"  Pod {i}: {len(models)} models")
        for model in models:
            print(f"    - {model}")
    
    print(f"\nStarting {actual_pods} pods...\n")
    
    # Run pods in parallel
    all_pod_results = []
    
    with ThreadPoolExecutor(max_workers=actual_pods) as executor:
        # Submit all pod workers
        futures = {}
        for pod_id, models in enumerate(work_distribution):
            future = executor.submit(run_pod_worker, pod_id, models, session_dir)
            futures[future] = pod_id
        
        # Track completion with progress bar
        with tqdm(total=actual_pods, desc="Pods") as pbar:
            for future in as_completed(futures):
                pod_id = futures[future]
                try:
                    result = future.result()
                    all_pod_results.append(result)
                    
                    if result['success']:
                        pbar.write(f"✓ Pod {pod_id} complete: {len(result['results'])} models ({result['elapsed_s']:.1f}s)")
                    else:
                        pbar.write(f"✗ Pod {pod_id} failed: {result['error']}")
                    
                except Exception as e:
                    pbar.write(f"✗ Pod {pod_id} exception: {e}")
                
                pbar.update(1)
    
    print(f"\n=== EXECUTION COMPLETE ===")
    
    # Summary
    successful_pods = sum(1 for r in all_pod_results if r['success'])
    failed_pods = len(all_pod_results) - successful_pods
    
    total_models_judged = sum(len([m for m in r['results'] if 'error' not in m or not m['error']]) 
                             for r in all_pod_results)
    
    print(f"\nPods successful: {successful_pods}/{actual_pods}")
    print(f"Pods failed: {failed_pods}")
    print(f"Models judged: {total_models_judged}")
    
    # Save results summary
    summary_file = session_dir / "summary.json"
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump({
            'session_dir': str(session_dir),
            'timestamp': datetime.now().isoformat(),
            'config': {
                'task_name': TASK_NAME,
                'judge_model': JUDGE_MODEL,
                'prompt_id': PROMPT_ID,
                'max_concurrent_pods': MAX_CONCURRENT_PODS,
                'actual_pods': actual_pods
            },
            'results': all_pod_results
        }, f, indent=2)
    
    print(f"\nSummary saved: {summary_file}")
    
else:
    print("\n✓ No models to judge - skipping execution")
    all_pod_results = []


=== PARALLEL EXECUTION ===
Session directory: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\runpod_llm_judge_logs\auto_v2_session_20251204_005123

Work Distribution:
  Pod 0: 10 models
    - anthropic__claude-sonnet-4.5
    - gemma3__12b
    - gemma3__27b
    - google__gemini-2.5-flash
    - llama3.2__3b
    - llama4__16x17b
    - mistral-small3.2__24b
    - openai__gpt-4.1
    - phi3__14b
    - phi4__14b
  Pod 1: 9 models
    - devstral__24b
    - gemma3__1b
    - gemma3__4b
    - llama3.2__1b
    - llama3.3__70b
    - mistral-large__123b
    - mixtral__8x22b
    - phi3.5__3.8b
    - phi4-mini__3.8b

Starting 2 pods...

[2025-12-04 00:51:23] [INFO] === Pod Session 0 Initialized ===
[2025-12-04 00:51:23] [INFO] === Pod Session 1 Initialized ===
[2025-12-04 00:51:23] [INFO] Models assigned: 10
[2025-12-04 00:51:23] [INFO] Models assigned: 9
[2025-12-04 00:51:23] [INFO] Creating RunPod instance...
[2025-12-04 00:51:23] [INFO] Creating RunPod instance...
[2025-12-04 00:51

Pods:   0%|          | 0/2 [00:00<?, ?it/s]

[2025-12-04 00:51:23] [INFO] ⚠ Pod creation failed on attempt 1: Something went wrong. Please try again later or contact support.
[2025-12-04 00:51:23] [INFO] Waiting 10 seconds before retrying...
raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'vwvsfws9q64vx8', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'mfy3z8h7umhg', 'machine': {'podHostId': 'vwvsfws9q64vx8-64410a32'}}}}
[2025-12-04 00:51:23] [INFO] Pod created: vwvsfws9q64vx8
[2025-12-04 00:51:23] [INFO] RunPod created: vwvsfws9q64vx8
[2025-12-04 00:51:23] [INFO] Waiting for SSH port (max 600s)...
[2025-12-04 00:51:24] [INFO] Attempt 1/60: 'NoneType' object has no attribute 'get'
[2025-12-04 00:51:33] [INFO] Attempt 2/3: creating pod...
raw_response: {'data': {'podFindAndDeployOnDemand': {'id': '3hyvta0oykf24k', 'imageName': 'runpod/pytorch

## Section 7: Validation & Summary

In [ ]:
if len(models_to_judge) > 0:
    print("\n=== FINAL VALIDATION ===")
    
    # Rebuild progress matrix
    print("\nRebuilding progress matrix...")
    final_progress_df = build_multi_judge_progress_matrix(
        PHASE4_ROOT,
        PHASE5_ROOT,
        TASK_NAME,
        JUDGE_MODEL,
        PROMPT_ID
    )
    
    print("\n=== FINAL PROGRESS MATRIX ===")
    display(final_progress_df)
    
    # Compare before/after
    initial_complete = len(complete_models)
    final_complete = len(final_progress_df[final_progress_df["status"] == "complete"])
    newly_complete = final_complete - initial_complete
    
    print(f"\n=== SUMMARY ===")
    print(f"Models complete (initial): {initial_complete}")
    print(f"Models complete (final): {final_complete}")
    print(f"Newly completed: {newly_complete}")
    
    # Check for failures
    still_incomplete = final_progress_df[final_progress_df["status"].isin(["partial", "not_started"])]
    if len(still_incomplete) > 0:
        print(f"\n⚠ Still incomplete: {len(still_incomplete)} models")
        for _, row in still_incomplete.iterrows():
            print(f"  - {row['model_name']}: {row['status']} ({row['judged_count']}/{row['total_samples']})")
    else:
        print(f"\n✓ All models complete!")
    
    print(f"\n→ Results saved to: {PHASE5_ROOT}")
    print(f"→ Logs saved to: {session_dir if 'session_dir' in locals() else LOG_DIR}")
    
else:
    print("\n✓ All models already complete - nothing to validate")

## Next Steps
1. **Review Results**: Check output files in Phase 5 directory
2. **Inspect Logs**: Review pod logs for any issues
3. **Re-run if needed**: Set `FORCE_REJUDGE=True` to re-judge specific models
4. **Analyze**: Use `llm-judge-parser.ipynb` to analyze judge outputs